In [ ]:
pip install pandas matplotlib seaborn scikit-learn


In [ ]:
#Task 1 — Data Exploration with Pandas

import pandas as pd

# Load data
df = pd.read_csv("students.csv")

# 1. basic inspection
print("--- First 5 Rows ---")
print(df.head())

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Summary Statistics ---")
print(df.describe())

# 2. Pass/Fail counts
print("\n--- Pass vs Fail Count ---")
print(df['passed'].value_counts())

# 3. Average per subject for Pass vs Fail
subject_cols = ['math', 'science', 'english', 'history', 'pe']
print("\n--- Avg Scores for Passing Students ---")
print(df[df['passed'] == 1][subject_cols].mean())

print("\n--- Avg Scores for Failing Students ---")
print(df[df['passed'] == 0][subject_cols].mean())

# 4. Find the top student
df['total_avg'] = df[subject_cols].mean(axis=1)
top_student = df.loc[df['total_avg'].idxmax()]
print(f"\nTop Student: {top_student['name']} with Average: {top_student['total_avg']:.2f}")



In [ ]:
#Task 2 & 3 — Visualization (Matplotlib & Seaborn)

import matplotlib.pyplot as plt
import seaborn as sns

# Prepare average score column
df['avg_score'] = df[subject_cols].mean(axis=1)

# --- Task 2: Matplotlib ---
# Plot 1: Bar Chart
plt.figure(figsize=(8,5))
df[subject_cols].mean().plot(kind='bar', color='skyblue')
plt.title("Average Score per Subject")
plt.ylabel("Score")
plt.savefig("plot1_bar.png")
plt.show()

# Plot 3: Scatter Plot (Study Hours vs Avg Score)
plt.figure(figsize=(8,5))
pass_students = df[df['passed'] == 1]
fail_students = df[df['passed'] == 0]
plt.scatter(pass_students['study_hours_per_day'], pass_students['avg_score'], color='green', label='Pass')
plt.scatter(fail_students['study_hours_per_day'], fail_students['avg_score'], color='red', label='Fail')
plt.title("Study Hours vs Average Score")
plt.xlabel("Hours per Day")
plt.ylabel("Avg Score")
plt.legend()
plt.savefig("plot3_scatter.png")
plt.show()

# --- Task 3: Seaborn ---
plt.figure(figsize=(10,5))
plt.subplot(1, 2, 1)
sns.barplot(data=df, x='passed', y='math')
plt.title("Math Score by Result")

plt.subplot(1, 2, 2)
sns.barplot(data=df, x='passed', y='science')
plt.title("Science Score by Result")
plt.tight_layout()
plt.savefig("seaborn_bar.png")
plt.show()

# COMMENT: Matplotlib is great for full control, but Seaborn 
# is much easier for comparing categories (like Pass/Fail) 
# because it handles the grouping and coloring automatically.

In [ ]:
#Task 4 — Machine Learning (Scikit-Learn)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Step 1: Prepare Data
X = df[['math', 'science', 'english', 'history', 'pe', 'attendance_pct', 'study_hours_per_day']]
y = df['passed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling (Standardizing data so all features have the same 'weight')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Train Model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Step 3: Evaluate
y_pred = model.predict(X_test_scaled)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100}%")

# Individual Predictions
test_names = df.loc[X_test.index, 'name'].values
for name, actual, pred in zip(test_names, y_test, y_pred):
    status = "✅" if actual == pred else "❌"
    print(f"{name}: Actual={actual}, Predicted={pred} {status}")

# Step 4: Feature Importance
coeffs = model.coef_[0]
features = X.columns
importance = sorted(zip(features, coeffs), key=lambda x: abs(x[1]), reverse=True)

print("\n--- Feature Importance ---")
for feat, val in importance:
    print(f"{feat}: {val:.4f}")

# Horizontal Bar Chart for Importance
plt.figure(figsize=(8,5))
colors = ['green' if c > 0 else 'red' for c in coeffs]
plt.barh(features, coeffs, color=colors)
plt.title("Feature Importance (Positive=Pass, Negative=Fail)")
plt.show()